In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

StateBackend

In [3]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

In [ ]:
# -----------------------------------------------------------------------------
# 1. Create the agent — these two are equivalent
# -----------------------------------------------------------------------------
agent = create_deep_agent(model="groq:llama-3.3-70b-versatile")

# Under the hood this is what `agent` is doing — explicit StateBackend:
agent2 = create_deep_agent(
    model="groq:llama-3.3-70b-versatile",
    backend=StateBackend(),
    
    
)

In [5]:
result=agent2.invoke({
    "messages":[{
        "role":"user",
        "content": ("Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it.")

    }]
})

In [6]:
# The agent's final natural-language reply
print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
I've created the file /notes/todo.txt with the specified content.


In [7]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2. Edit video\n3. Upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='f6e202f8-62cd-4df5-8652-a44d7614c0d0'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jvzkqbmfy', 'function': {'arguments': '{"content":"1. Record video\\n2. Edit video\\n3. Upload video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 9161, 'total_tokens': 9213, 'completion_time': 0.175107194, 'completion_tokens_details': None, 'prompt_time': 0.468845661, 'prompt_tokens_details': None, 'queue_time': 0.170989129, 'total_time': 0.643952855}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e9

In [8]:
print("\n--- Backend check -----------------------------------------------")
files=result.get("files",[])

if files:
    print(f"✅ StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")        



--- Backend check -----------------------------------------------
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-06-06T10:31:51.492317+00:00', 'modified_at': '2026-06-06T10:31:51.492317+00:00'}


In [9]:
# -----------------------------------------------------------------------------
# 4. Prove persistence WITHIN the same thread:
#    feed the returned state back in and ask it to READ the file
# -----------------------------------------------------------------------------
followup = agent2.invoke({
    # carry forward prior messages + the files state
    "messages": result["messages"] + [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me ."
    }],
    "files": result.get("files", {}),   # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) -------------------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) -------------------------------------
The contents of /notes/todo.txt are:
1. Record video
2. Edit video
3. Upload video


FileSystemBackend(Local Disk)

In [10]:
# -----------------------------------------------------------------------------
# 1. Create the agent with a real-disk backend
#    root_dir="." -> files land relative to your current working directory
#    virtual_mode=True -> agent uses virtual paths like /notes/todo.txt,
#                         mapped onto root_dir
# -----------------------------------------------------------------------------
from deepagents.backends import FilesystemBackend
ROOT = "."

agent=create_deep_agent(model="groq:llama-3.3-70b-versatile",backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))
print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


In [11]:
# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
# -----------------------------------------------------------------------------
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
I've created the file /notes/todo.txt with the specified content.


In [15]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working — look on the REAL disk
#    With virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt
# -----------------------------------------------------------------------------
from pathlib import Path
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 50}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 C:\Deep Agent\DeepAgents\notes\todo.txt
--------------------------------------------------
1. Record video
2. Edit video
3. Upload video


In [21]:
# -----------------------------------------------------------------------------
# 4. Prove persistence ACROSS sessions:
#    Unlike StateBackend, this file survives even after Python exits.
#    A brand-new agent (fresh state) can read it back from disk.
# -----------------------------------------------------------------------------
fresh_agent = create_deep_agent(
    model="groq:llama-3.3-70b-versatile",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01krp3t1faftvt3knt2dfgfe3r` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 96720, Requested 9137. Please try again in 1h24m20.447999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

StoreBackend 

In [9]:
from langgraph.store.memory import InMemoryStore
from deepagents import create_deep_agent
from deepagents.backends import StoreBackend
store=InMemoryStore()

agent=create_deep_agent(
    model="groq:openai/gpt-oss-120b",
    backend=StoreBackend(

        namespace=lambda rt:("demo-user",),
    ),
    store=store
)

print("✅ Agent created with StoreBackend using an InMemoryStore.")

✅ Agent created with StoreBackend using an InMemoryStore.


In [10]:
import os
import uuid
# -----------------------------------------------------------------------------
# 2. THREAD 1 — write a file
# -----------------------------------------------------------------------------
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
I've created /notes/todo.txt with the requested content.


In [ ]:
# --- Thread 2: read back on a DIFFERENT thread ---------------------------
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)

In [11]:
# --- Thread 2: read back on a DIFFERENT thread ---------------------------
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
1. Record video
2. Edit video
3. Upload video


# Deep Agent Backends — Where Your Files Actually Live

Every Deep Agent works with a **virtual filesystem**. Its tools read and write files using paths such as:

```text
/notes/todo.txt
```

However, those paths are only an abstraction. The **backend** determines where the data is actually stored. This single choice affects persistence, sharing, and durability.

The agent code remains the same across all backends. Only the backend configuration changes.

---

# 1. StateBackend (Default)

Files live inside the agent's **LangGraph state** (in memory) and are tied to a single thread. They are available through `result["files"]`, and you must manually pass that state forward if you want to continue using those files.

### Characteristics

* **Where:** In-memory dictionary inside a thread's state
* **Cross-thread access:** ❌ No
* **Survives process restart:** ❌ No
* **Real files on disk:** ❌ No

### Best For

* Temporary working files
* Scratch space
* Short-lived agent workflows

### Example

```python
agent = create_deep_agent(
    model="groq:llama-3.3-70b-versatile"
)
```

---

# 2. FilesystemBackend

Files are written to your actual filesystem under a specified `root_dir`.

With `virtual_mode=True`, a virtual path such as:

```text
/notes/todo.txt
```

is mapped to:

```text
root_dir/notes/todo.txt
```

These are genuine files that can be opened in an editor, viewed in a terminal, or committed to source control.

### Characteristics

* **Where:** Your local filesystem
* **Cross-thread access:** ✅ Yes
* **Survives process restart:** ✅ Yes
* **Real files on disk:** ✅ Yes

### Best For

* Editing project files
* Code generation workflows
* Persistent file storage

### Example

```python
from deepagents.backends import FilesystemBackend

agent = create_deep_agent(
    model="groq:llama-3.3-70b-versatile",
    backend=FilesystemBackend(
        root_dir="./workspace",
        virtual_mode=True
    )
)
```

> ⚠️ **Caution:** The agent receives real read/write access to the specified directory. Always use a dedicated workspace or sandbox directory.

---

# 3. StoreBackend

Files are stored inside a **LangGraph Store** and scoped by a namespace.

This is the only backend that allows files to be shared across different threads and conversations when they use the same store and namespace.

Using `InMemoryStore` provides persistence only while the process is running. Replacing it with a persistent store (such as a PostgreSQL-backed store) enables true long-term durability.

### Characteristics

* **Where:** LangGraph Store
* **Cross-thread access:** ✅ Yes
* **Survives process restart:** ⚠️ Only with a persistent store
* **Real files on disk:** ❌ No

### Best For

* Long-term memory
* Per-user file spaces
* Shared storage across conversations

### Example

```python
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend

store = InMemoryStore()

agent = create_deep_agent(
    model="groq:llama-3.3-70b-versatile",
    backend=StoreBackend(
        namespace=lambda rt: ("demo-user",)
    ),
    store=store
)
```

### Cross-Thread Example

**Thread 1**

```text
Create /notes/todo.txt
```

**Thread 2**

```text
Read /notes/todo.txt
```

As long as both threads use the same store and namespace, the file is accessible from both.

---

# Backend Comparison

| Backend           | Lives In         | Cross-Thread? | Survives Restart?             | Real File on Disk? |
| ----------------- | ---------------- | ------------- | ----------------------------- | ------------------ |
| StateBackend      | LangGraph State  | ❌ No          | ❌ No                          | ❌ No               |
| FilesystemBackend | Local Filesystem | ✅ Yes         | ✅ Yes                         | ✅ Yes              |
| StoreBackend      | LangGraph Store  | ✅ Yes         | ⚠️ Only with Persistent Store | ❌ No               |

---

# Mental Model

The agent never knows which backend is being used.

It always interacts with files through virtual paths such as:

```text
/notes/todo.txt
```

The backend silently translates every read and write operation into the appropriate storage mechanism:

* **StateBackend** → Thread state
* **FilesystemBackend** → Local disk
* **StoreBackend** → LangGraph store

This abstraction allows you to switch storage strategies without changing any agent logic.
